# ChromaDB

**Chroma** is an open-source, developer-first **embedding database** (vector store). You hand it documents and embeddings; it stores them, indexes the vectors for approximate nearest-neighbor search, and answers similarity queries with metadata filtering — all from a few lines of Python, with no separate server to run.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**What:** ChromaDB is a vector database built for AI applications, especially **retrieval-augmented generation (RAG)**. Its core unit is a **collection**: a named bucket of records, each holding an `id`, a vector `embedding`, the original `document` text, and arbitrary `metadata`. You query a collection with a vector (or raw text, which Chroma embeds for you) and get back the *k* nearest records, optionally filtered by metadata.

**The problem it solves:** LLMs have a fixed context window and a frozen knowledge cutoff. To ground them in your own data you embed your documents once, store the vectors, and at query time retrieve the few chunks most semantically similar to the user's question and stuff them into the prompt. Doing that needs fast similarity search over thousands-to-millions of vectors plus metadata filtering — exactly what Chroma packages.

**Why reach for it:** It optimizes for **time-to-first-prototype**. `pip install chromadb`, one `EphemeralClient()`, and you have a working vector store with a built-in embedding model — no Docker, no cluster, no API key. It bundles the embedding step, the index, and the storage so a RAG demo fits on one screen.

**When not to:** At very large scale (tens of millions of vectors, high write throughput, multi-node sharding) or when you need it managed/serverless with strong SLAs, reach for Pinecone, Weaviate, Qdrant, Milvus, or `pgvector`. Chroma shines for local dev, notebooks, small-to-medium apps, and single-node deployments.

## 2. Mental Model

Think of Chroma as **SQLite for embeddings**: an embedded library that runs *inside* your process and persists to a local directory, not a server you connect to over the network. The mental hierarchy is small:

```
Client  (EphemeralClient = in-RAM, PersistentClient = on-disk, HttpClient = remote server)
  └── Collection  ("documents", "products", ...)  — has one distance metric + one embedding fn
        └── Records:  id | embedding (vector) | document (text) | metadata (dict)
```

The key insight: **a collection owns its embedding function.** When you `add(documents=[...])` *without* embeddings, Chroma calls the collection's embedding function to turn text into vectors for you; when you `query(query_texts=[...])`, it embeds the query the same way. If you pass `embeddings=[...]` directly, Chroma stores them verbatim and never calls a model — that is the fully-offline path. Same model in, same model out: never mix embedding spaces within a collection.

## 3. Key Concepts

- **Client** — your handle to the database. `EphemeralClient()` keeps everything in RAM (great for tests/notebooks); `PersistentClient(path=...)` writes to a local DuckDB/SQLite + Parquet store and survives restarts; `HttpClient(host, port)` talks to a `chroma run` server.
- **Collection** — a namespace of records sharing one **distance metric** and one **embedding function**. Created with `create_collection` / fetched with `get_or_create_collection`.
- **Embedding function** — maps text → vector. The default is `all-MiniLM-L6-v2` (384-dim) via ONNX, downloaded on first use. You can swap in OpenAI, Cohere, Sentence-Transformers, etc., or skip it entirely by supplying your own `embeddings`.
- **Distance / space** — set via `metadata={"hnsw:space": "cosine"}` at collection creation: `l2` (default, squared Euclidean), `cosine`, or `ip` (inner product). Returned **distances are smaller = more similar**; cosine distance is `1 - cosine_similarity`.
- **Metadata & `where` filter** — each record carries a JSON-ish dict; queries can filter with Mongo-style operators (`{"year": {"$gte": 2020}}`, `{"$and": [...]}`). `where_document={"$contains": "text"}` filters on the raw document.
- **HNSW index** — the underlying ANN algorithm (Hierarchical Navigable Small World graph). Approximate, fast, tunable via `hnsw:*` params. Results are not guaranteed exact at scale.
- **Core ops** — `add`, `upsert`, `get`, `update`, `delete`, `query`, `count`, `peek`.

## 4. Setup

Pure-Python install, CPU-friendly:

```bash
pip install chromadb
```

That single wheel includes an embedded HNSW index and bundles `onnxruntime` so the default `all-MiniLM-L6-v2` embedding model works out of the box (it downloads the ~80 MB model on first use). For a standalone server instead of an embedded client, run `chroma run --path ./db` and connect with `chromadb.HttpClient()`.

The examples below avoid that download by passing **explicit embeddings**, so they run fully offline.

In [1]:
# Confirm the install and show the version. No network needed for this cell.
import chromadb

print("chromadb", chromadb.__version__)

chromadb 1.5.9


## 5. Worked Examples

### Example 1 — Build a collection and run a similarity query (offline)

We pass our own tiny 3-dimensional vectors so nothing downloads. The three axes stand in for crude "topics" (animals / finance / filler). In real code you'd let an embedding model produce these vectors.

In [2]:
import chromadb

client = chromadb.EphemeralClient()  # in-RAM; perfect for notebooks and tests

# Cosine space so distance = 1 - cosine_similarity.
col = client.create_collection("articles", metadata={"hnsw:space": "cosine"})

docs = [
    "cats and kittens love to nap",
    "puppies and dogs at the park",
    "the stock market fell sharply today",
    "central banks raised interest rates",
]
# Hand-built 3-dim embeddings: [animal, finance, filler].
embeddings = [
    [1.0, 0.0, 0.1],
    [0.9, 0.0, 0.2],
    [0.0, 1.0, 0.1],
    [0.0, 0.9, 0.2],
]
col.add(
    ids=["a1", "a2", "a3", "a4"],
    documents=docs,
    embeddings=embeddings,
    metadatas=[
        {"topic": "animals", "year": 2021},
        {"topic": "animals", "year": 2023},
        {"topic": "finance", "year": 2022},
        {"topic": "finance", "year": 2024},
    ],
)
print("records in collection:", col.count())

# Query with an "animal-ish" vector -> expect the two animal docs back.
res = col.query(query_embeddings=[[1.0, 0.0, 0.0]], n_results=2)
for doc, dist in zip(res["documents"][0], res["distances"][0]):
    print(f"  dist={dist:.4f}  {doc}")

records in collection: 4
  dist=0.0050  cats and kittens love to nap
  dist=0.0238  puppies and dogs at the park


Lower distance = more similar. The two animal documents come back first, the finance ones are far away — exactly what a real RAG retrieve step does, just with semantic embeddings instead of our toy vectors.

### Example 2 — Metadata filtering, updates, and on-disk persistence

`where` narrows the search to records matching a metadata predicate *before* ranking. We also show `upsert`/`update`/`delete` and a `PersistentClient` that survives a restart.

In [3]:
import os, tempfile, chromadb

# --- metadata filtering on the existing in-RAM collection ---
res = col.query(
    query_embeddings=[[0.5, 0.5, 0.0]],          # ambiguous query...
    n_results=2,
    where={"$and": [{"topic": "finance"}, {"year": {"$gte": 2023}}]},  # recent finance only
)
print("filtered hits:", res["documents"][0])

# --- mutate records ---
col.upsert(ids=["a1"], documents=["cats, kittens, and tabbies"], embeddings=[[1.0, 0.0, 0.0]])
col.update(ids=["a2"], metadatas=[{"topic": "animals", "year": 2025}])
col.delete(ids=["a4"])
print("count after upsert/delete:", col.count())

# --- persistence: write to disk, reopen, confirm data survived ---
path = os.path.join(tempfile.mkdtemp(), "chroma_db")
pclient = chromadb.PersistentClient(path=path)
p = pclient.get_or_create_collection("notes")
p.add(ids=["n1"], documents=["remember to persist"], embeddings=[[0.1, 0.2, 0.3]])

reopened = chromadb.PersistentClient(path=path).get_collection("notes")
print("reopened from disk, count:", reopened.count())

filtered hits: ['central banks raised interest rates']
count after upsert/delete: 3
reopened from disk, count: 1


### Optional — the built-in embedding function (gated, downloads a model)

Drop the explicit vectors and Chroma embeds text for you with `all-MiniLM-L6-v2`. This downloads ~80 MB on first run, so it's gated behind an env var; the call *shape* is shown either way.

In [4]:
import os, chromadb

if os.getenv("CHROMA_RUN_EMBED"):
    client = chromadb.EphemeralClient()
    col2 = client.create_collection("auto")  # uses the default embedding function
    # No embeddings= -> Chroma embeds the documents for you.
    col2.add(ids=["1", "2"], documents=["a fluffy kitten", "quarterly earnings report"])
    # query_texts (not query_embeddings) -> the query is embedded the same way.
    hit = col2.query(query_texts=["a small cat"], n_results=1)
    print("semantic hit:", hit["documents"][0])
else:
    print("Set CHROMA_RUN_EMBED=1 to run the default-embedding example (downloads ~80MB).")
    print("Shape: col.add(ids=[...], documents=[...]); col.query(query_texts=[...], n_results=k)")

Set CHROMA_RUN_EMBED=1 to run the default-embedding example (downloads ~80MB).
Shape: col.add(ids=[...], documents=[...]); col.query(query_texts=[...], n_results=k)


## 6. Gotchas & Pitfalls

- **Mixing embedding models within a collection.** A collection's vectors must all come from the same model/dimension. If you `add` with the default function and later `query` with OpenAI embeddings, the geometry is meaningless. Pin one embedding function per collection.
- **Default model downloads on first use.** The bundled `all-MiniLM-L6-v2` fetches ~80 MB the first time — surprising in air-gapped CI. Pre-cache it, vendor the model, or pass explicit `embeddings`.
- **`add` vs `upsert`.** `add` raises (or warns) on duplicate IDs; `upsert` overwrites. For idempotent pipelines that may re-run, prefer `upsert`.
- **Distance ≠ similarity.** `query` returns **distances** where *smaller is closer*. With cosine space that's `1 - cosine_similarity`; with the default `l2` it's *squared* Euclidean. Don't sort the wrong way.
- **Metric is fixed at creation.** You set `hnsw:space` when the collection is created and can't change it later — recreate the collection to switch metrics.
- **`results["documents"]` is a list-of-lists.** `query` is batched over queries, so a single query's hits live at `res["documents"][0]`, not `res["documents"]`. Same for `distances`, `metadatas`, `ids`.
- **Metadata values are scalars only.** Strings, ints, floats, bools — no nested dicts or lists. Flatten before storing.
- **HNSW is approximate.** At scale, top-*k* may miss a true neighbor. Raise `n_results` and tune `hnsw:search_ef` if recall matters; for tiny collections it's effectively exact.
- **Persistence is per-`path`.** Two `PersistentClient`s on different paths see different data; an `EphemeralClient` vanishes when the process exits.

## 7. When to Use vs Alternatives

**Reach for Chroma when:** you're prototyping RAG, building a local/single-node app, working in notebooks, or want the embedding + index + storage stack in one `pip install` with zero infra. It's the fastest path from "I have documents" to "I can retrieve them."

| Option | Sweet spot | Trade-off vs Chroma |
|---|---|---|
| **Chroma** | Local dev, notebooks, small–medium single-node RAG | Easiest start; not built for massive scale / multi-node |
| **FAISS** | Max-control, in-process ANN over huge vector sets | A *library*, not a DB — no metadata store, persistence, or filtering; you build those (see the FAISS notebook) |
| **pgvector** | You already run Postgres and want vectors beside relational data | One system, transactional; ANN less specialized, manual tuning |
| **Qdrant / Weaviate / Milvus** | Self-hosted, production-scale, rich filtering & sharding | More features and scale; heavier ops (server/cluster) |
| **Pinecone** | Fully-managed, serverless, hands-off scaling | No infra to run; SaaS cost + vendor lock-in, needs API key |

**Rule of thumb:** start in Chroma to validate the retrieval quality and prompt, then graduate to a managed or clustered store only when scale, throughput, or SLA demands force the move. The retrieval *concepts* (embeddings, distance metric, top-*k*, metadata filter) carry over unchanged.

## 8. Resources

- **Official docs** — https://docs.trychroma.com/
- **Getting started guide** — https://docs.trychroma.com/docs/overview/getting-started
- **GitHub repo** — https://github.com/chroma-core/chroma
- **Embedding functions reference** — https://docs.trychroma.com/integrations/embedding-models
- **Collections & query API** — https://docs.trychroma.com/docs/collections/manage-collections

**Cross-links:** for the raw ANN library underneath these vector DBs see the **FAISS** notebook; for where these embeddings come from see **Vector Embeddings**; for the end-to-end pipeline that consumes retrieved chunks see **Retrieval-Augmented Generation (RAG)**.